<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-04-rag/lesson-4.3-rag-engine/notebooks/GCP_Capstone_4.3_RAG_Engine.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.3 Vertex AI RAG Engine — Managed RAG Pipeline
**Netsetos GenAI Engineering — GCP Capstone**

Create corpus, import from GCS/Drive/Slack/Jira, retrieve, and generate with automatic citations.


## Setup


In [ ]:
!pip install -q google-cloud-aiplatform google-genai==2.21.0
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from vertexai import rag
import vertexai
from google import genai
from google.genai import types

vertexai.init(project=PROJECT_ID, location='us-central1')
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # Gemini 3.x generation: global (corpus stays regional)


## Prepare a new project for serverless RAG Engine (one-time)
A brand-new project needs three things before `create_corpus` works, all handled by the cell below (idempotent — safe to re-run):
1. **APIs enabled** — `vectorsearch.googleapis.com` backs the serverless vector store and is **off by default**; a missing enable is exactly what makes `create_corpus` fail with *“Vector Search API … has not been used … or it is disabled”*. (`aiplatform` + `storage` are enabled too.)
2. **Serverless mode** — the default “Scaled”/Spanner store is allowlist-only for new projects; Serverless mode is open to everyone. Serverless (Vector Search 2.0) has no fixed base fee: vector storage and queries are billed per use — verify on the Vertex AI pricing page (2026-09-03).
3. **Propagation** — API enablement is eventually consistent, so the first corpus is created through `create_corpus_ready(...)`, which retries while enablement propagates.

> Serverless RAG corpora (Vector Search 2.0) are **`us-central1`-only** — not `asia-south1`; if India data residency is a requirement, the DIY Firestore path (4.2) is the `asia-south1` option, as 4.4 says. The corpus, embedding and retrieval client stays regional; only the Gemini-3.x generation client uses `global`.

In [ ]:
# One-time new-project prep for serverless RAG Engine. Idempotent — safe to re-run.
import google.auth, google.auth.transport.requests, requests, subprocess, time
from google.api_core import exceptions as _gexc
_RAG_LOCATION = 'us-central1'   # serverless RAG corpora (Vector Search 2.0) are us-central1-only

# 1) Enable the APIs a fresh project needs. vectorsearch backs the serverless vector store
#    (off by default -> the create_corpus 403); storage is for GCS import.
subprocess.run(['gcloud', 'services', 'enable',
                'aiplatform.googleapis.com', 'vectorsearch.googleapis.com',
                'storage.googleapis.com', '--project', PROJECT_ID], check=False)

# 2) Switch RAG Engine to Serverless mode (open to everyone, no allowlist).
_creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
_creds.refresh(google.auth.transport.requests.Request())
_r = requests.patch(
    f'https://{_RAG_LOCATION}-aiplatform.googleapis.com/v1beta1/'
    f'projects/{PROJECT_ID}/locations/{_RAG_LOCATION}/ragEngineConfig',
    headers={'Authorization': f'Bearer {_creds.token}'},
    json={'ragManagedDbConfig': {'serverless': {}}}, timeout=60)
print('Serverless mode:', 'ready' if _r.ok else f'{_r.status_code} {_r.text[:150]}')

# 3) create_corpus wrapper that waits out API-enablement propagation on fresh projects.
def create_corpus_ready(**kwargs):
    for _attempt in range(8):  # ~ up to 4 min
        try:
            return rag.create_corpus(**kwargs)
        except (_gexc.PermissionDenied, _gexc.FailedPrecondition) as e:
            m = str(e).lower()
            if 'has not been used' in m or 'is disabled' in m or 'service_disabled' in m:
                print('vectorsearch API still propagating; retrying in 30s...'); time.sleep(30); continue
            if 'allowlist' in m or 'restricted' in m or 'not allowed' in m:
                print('the Serverless switch above has not propagated yet (the default store is allowlist-only); retrying in 30s...'); time.sleep(30); continue
            raise
    raise RuntimeError('vectorsearch.googleapis.com still not ready — wait 1-2 min and re-run.')
print('APIs enabled. First corpus uses create_corpus_ready(...) to ride out enablement propagation.')

## Create a docs bucket (named from your project) and put DocuMind's corpus in it
The lesson imports documents from Cloud Storage. This creates a bucket `{PROJECT_ID}-rag-docs` in `us-central1` and copies `deploy/evals/corpus/acme` into it: the tenant's synthetic handbook, MSA, invoice and report as markdown, and the **thirteen real documents** as PDFs — RAG Engine parses PDFs with its own layout parser, so the scanned POSH Gazette goes in too. Idempotent — safe to re-run.


In [ ]:
# Create a GCS bucket for this lesson's documents (named from the project so it is
# globally unique) and put DocuMind's corpus in it. Idempotent — safe to re-run.
import os, subprocess
BUCKET = f'{PROJECT_ID}-rag-docs'
_exists = subprocess.run(['gcloud', 'storage', 'buckets', 'describe', f'gs://{BUCKET}',
                          '--project', PROJECT_ID], capture_output=True, text=True).returncode == 0
if not _exists:
    subprocess.run(['gcloud', 'storage', 'buckets', 'create', f'gs://{BUCKET}',
                    '--project', PROJECT_ID, '--location', 'us-central1',
                    '--uniform-bucket-level-access'], check=True)

# The documents: deploy/evals/corpus/acme from the course repo (beside this notebook, or cloned
# under /content the way 10.4 does). The thirteen real documents (deploy/evals/real_sources.json) go up as
# PDFs - RAG Engine runs its own layout parser over them, scanned Gazette included - and the
# tenant's synthetic handbook, MSA, invoice and report as markdown. A .md that mirrors a .pdf
# stays home: it exists for the offline gate, and uploading it would index every Act twice.
def corpus_dir(tenant='acme'):
    for base in ('deploy/evals/corpus', '../deploy/evals/corpus', '/content/agentic-ai-weekend-gcp/deploy/evals/corpus'):
        if os.path.isdir(f'{base}/{tenant}'):
            return f'{base}/{tenant}'
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'feat/lesson-4.8-live-evals', 'https://github.com/netsetos/agentic-ai-weekend-gcp',
                    '/content/agentic-ai-weekend-gcp'], check=True)
    return f'/content/agentic-ai-weekend-gcp/deploy/evals/corpus/{tenant}'

_dir = corpus_dir()
_files = [f for f in sorted(os.listdir(_dir))
          if not (f.endswith('.md') and os.path.exists(os.path.join(_dir, f[:-3] + '.pdf')))]
subprocess.run(['gcloud', 'storage', 'cp', *[os.path.join(_dir, f) for f in _files],
                f'gs://{BUCKET}/docs/', '--project', PROJECT_ID], check=True)
print(f'Docs bucket ready: gs://{BUCKET}/docs/  ({len(_files)} files)')
for f in _files:
    print('  ', f)


## Cell 1: Create a RAG Corpus


In [ ]:
embedding_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model='publishers/google/models/text-embedding-005'
    )
)

# Look it up first: create_corpus does not de-duplicate on display_name, so a re-run would leave a
# second (billed) corpus behind. One name, one corpus, however many times the cell runs.
CORPUS_DISPLAY_NAME = 'documind-lesson43'
corpus = next((c for c in rag.list_corpora() if c.display_name == CORPUS_DISPLAY_NAME), None)
if corpus:
    print(f'Corpus exists - reusing: {corpus.name}')
else:
    corpus = create_corpus_ready(
        display_name=CORPUS_DISPLAY_NAME,
        description='Lesson 4.3 test corpus',
        backend_config=rag.RagVectorDbConfig(
            rag_embedding_model_config=embedding_config),
    )
    print(f'Corpus: {corpus.name}')

# List corpora
for c in rag.list_corpora():
    print(f'  {c.display_name}: {c.name}')


## Cell 2: Import from GCS


In [ ]:
# Grant the RAG service agent read on the docs bucket (create_corpus above provisioned
# it); without this grant, import returns 0 files silently.
import subprocess
_pn = subprocess.run(['gcloud', 'projects', 'describe', PROJECT_ID, '--format=value(projectNumber)'],
                     capture_output=True, text=True).stdout.strip()
_grant = subprocess.run(['gcloud', 'storage', 'buckets', 'add-iam-policy-binding', f'gs://{BUCKET}',
                         '--member', f'serviceAccount:service-{_pn}@gcp-sa-vertex-rag.iam.gserviceaccount.com',
                         '--role', 'roles/storage.objectViewer', '--project', PROJECT_ID],
                        capture_output=True, text=True, check=False)
print('bucket grant:', 'ok' if _grant.returncode == 0 else _grant.stderr.strip()[-200:])   # silent failure = 0 files imported

response = rag.import_files(
    corpus.name,
    [f'gs://{BUCKET}/docs/'],
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512, chunk_overlap=100)),
    max_embedding_requests_per_min=900,
)
print(f'Imported: {response.imported_rag_files_count}')
print(f'Skipped: {response.skipped_rag_files_count}')
if response.imported_rag_files_count == 0:
    print('WARNING: 0 files imported. The service-agent grant above may still be '
          'propagating (~1-2 min) — wait and re-run this cell.')

## Cell 3: Import from Google Drive


In [ ]:
# OPTIONAL — import from a Google Drive folder (Cell 2's GCS path already showed ingestion).
# Drive folders are your own, so this one is bring-your-own:
#   1. Put files in a Drive folder; copy its id from the URL
#      (drive.google.com/drive/folders/<THIS_PART>).
#   2. Share that folder as Viewer with the RAG service agent printed below.
#   3. Set DRIVE_FOLDER_ID and re-run.
import subprocess
_pn = subprocess.run(['gcloud', 'projects', 'describe', PROJECT_ID, '--format=value(projectNumber)'],
                     capture_output=True, text=True).stdout.strip()
print(f'Share your Drive folder (Viewer) with: service-{_pn}@gcp-sa-vertex-rag.iam.gserviceaccount.com')

DRIVE_FOLDER_ID = 'YOUR_FOLDER_ID'  # CHANGE, or leave as-is to skip
if DRIVE_FOLDER_ID == 'YOUR_FOLDER_ID':
    print('Skipping Drive import — set DRIVE_FOLDER_ID to your own folder id (shared above) to run.')
else:
    drive_response = rag.import_files(
        corpus.name,
        [f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}'],
        transformation_config=rag.TransformationConfig(
            chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)),
        max_embedding_requests_per_min=900)
    print(f'Drive import: {drive_response.imported_rag_files_count}')

## Cell 4: Direct Retrieval


In [ ]:
response = rag.retrieval_query(
    rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
    text='After how many years of continuous service does gratuity become payable?',   # s. 4, Payment of Gratuity Act, 1972
    rag_retrieval_config=rag.RagRetrievalConfig(
        top_k=5,
        filter=rag.Filter(vector_distance_threshold=0.5)),
)

for ctx in response.contexts.contexts:
    print(f'Source: {ctx.source_uri}')
    print(f'Score: {ctx.score:.3f}  (cosine distance, lower = more relevant)')
    print(f'Text: {ctx.text[:150]}...\n')


## Cell 5: Grounded Generation


In [ ]:
rag_retrieval_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[types.VertexRagStoreRagResource(rag_corpus=corpus.name)],
            rag_retrieval_config=types.RagRetrievalConfig(
                top_k=5,
                filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Which Acts does the Code on Wages, 2019 repeal, and what is its overtime rate?',
    config=types.GenerateContentConfig(tools=[rag_retrieval_tool]))
print(response.text)

# Grounding metadata — automatic citations
for candidate in response.candidates:
    gm = candidate.grounding_metadata
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.retrieved_context.uri}')
    if gm and gm.grounding_supports:
        for support in gm.grounding_supports:
            print(f'  Claim: {support.segment.text}')
            print(f'  Backed by: {support.grounding_chunk_indices}')


## Cell 6: The ManagedRAG module (outside the kit by design)


In [ ]:
# DocuMind managed RAG (Vertex AI RAG Engine) - the lesson's module; outside the kit by design, the lane runs the DIY path of 4.2 and 4.5
class ManagedRAG:
    def __init__(self, project, location="us-central1"):
        vertexai.init(project=project, location=location)
        self.client = genai.Client(enterprise=True, project=project, location="global")  # generation: global (corpus stays regional)
        self.corpus = None

    def create_corpus(self, name, description=""):
        emb = rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model="publishers/google/models/text-embedding-005"))
        self.corpus = rag.create_corpus(
            display_name=name, description=description,
            backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=emb))
        return self.corpus.name

    def use_corpus(self, corpus_name):
        self.corpus = rag.get_corpus(name=corpus_name)

    def ingest(self, paths, chunk_size=512, chunk_overlap=100, layout_parser=None):
        kwargs = {"corpus_name": self.corpus.name, "paths": paths,
                  "transformation_config": rag.TransformationConfig(
                      chunking_config=rag.ChunkingConfig(chunk_size=chunk_size, chunk_overlap=chunk_overlap)),
                  "max_embedding_requests_per_min": 900}
        if layout_parser:
            kwargs["layout_parser"] = layout_parser
        return rag.import_files(**kwargs)

    def retrieve(self, query, top_k=5, threshold=0.5):
        return rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=self.corpus.name)],
            text=query,
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=top_k, filter=rag.Filter(vector_distance_threshold=threshold)))

    def ask(self, question, model_name="gemini-3.6-flash", top_k=5):
        rag_tool = types.Tool(retrieval=types.Retrieval(
            vertex_rag_store=types.VertexRagStore(
                rag_resources=[types.VertexRagStoreRagResource(rag_corpus=self.corpus.name)],
                rag_retrieval_config=types.RagRetrievalConfig(
                    top_k=top_k, filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))
        return self.client.models.generate_content(
            model=model_name, contents=question,
            config=types.GenerateContentConfig(tools=[rag_tool]))

print('ManagedRAG class ready')


## Cell 7: Cleanup


In [ ]:
# Delete the corpus when done: RAG Engine storage bills while it exists, and it does not expire.
# Cell 1 finds it again by display_name, so leaving it costs storage but never a duplicate.
# rag.delete_corpus(name=corpus.name)
# print('Corpus deleted')


## ✅ Lesson 4.3 Complete!

- ✅ Created RAG corpus with text-embedding-005
- ✅ Imported from GCS (and optionally Drive)
- ✅ Automatic chunking + embedding + indexing
- ✅ Direct retrieval with retrieval_query()
- ✅ Grounded generation with automatic citations
- ✅ The ManagedRAG module, the managed alternative to the lane's DIY path

**Three of Module 4's seven lessons done:**
- 4.1: Document AI ingestion (OCR/Layout/Form)
- 4.2: DIY RAG pipeline (manual embed→retrieve→generate)
- 4.3: Managed RAG Engine (corpus→import→query)

Module 4 in one line: 4.1 Document AI · 4.2 DIY RAG · 4.3 RAG Engine · 4.4 Vertex AI Search + Google Search grounding · 4.5 Context Engineering · 4.6 Graph RAG · 4.7 Evaluation.

**Next: 4.4 — Vertex AI Search and Google Search grounding (the fourth tier), then 4.5 Context Engineering**
